[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/notebooks/20_PCA.ipynb)

# Componentes principales (PCA)

*¿Para qué se utiliza?*
El Análisis de Componentes Principales (PCA, por sus siglas en inglés) es una técnica de reducción de dimensiones: busca combinaciones lineales de un conjunto de variables numéricas correlacionadas —los *componentes principales*— que concentran la mayor cantidad posible de la variabilidad original en unas pocas dimensiones. A diferencia del análisis factorial (notebook 21), PCA no asume la existencia de variables latentes que "causan" las correlaciones observadas; simplemente busca la combinación de variables que explica la mayor varianza posible, sin más supuesto que ese.

Ejemplos de uso en negocios:
- Un analista tiene 10 indicadores financieros muy correlacionados entre sí y quiere resumirlos en 2 o 3 componentes antes de usarlos como predictores en un modelo de regresión, evitando problemas de multicolinealidad.
- Un equipo de producto tiene decenas de especificaciones técnicas de una línea de vehículos o electrodomésticos y quiere resumirlas para comparar visualmente los modelos entre sí.
- Un área de riesgo necesita reducir un conjunto grande de variables macroeconómicas a unos pocos factores clave antes de modelar el riesgo de una cartera.

*Variables consideradas*
Variables cuantitativas continuas, idealmente con correlaciones moderadas o altas entre sí — si las variables no están correlacionadas, PCA no logra reducir la dimensionalidad de forma útil (cada componente explicaría aproximadamente la misma proporción de varianza).

*¿Cómo funciona?*
PCA transforma las variables originales en un nuevo conjunto de variables (los componentes), no correlacionadas entre sí, ordenadas de mayor a menor varianza explicada. El primer componente (PC1) es la combinación lineal que captura la mayor varianza posible; el segundo componente (PC2) captura la mayor varianza posible entre las combinaciones no correlacionadas con PC1, y así sucesivamente. El número de componentes es igual al número de variables originales, pero típicamente solo los primeros 2 o 3 concentran la mayor parte de la información.

*Supuestos y recomendaciones*
- No requiere variable dependiente, normalidad ni homocedasticidad: es una técnica exploratoria.
- Es indispensable **estandarizar las variables** antes de aplicar PCA (media 0, desviación estándar 1), ya que la técnica es sensible a la escala: una variable medida en miles dominaría artificialmente los componentes frente a una variable medida en unidades.
- Es más útil cuanto mayor sea la correlación entre las variables originales; si la correlación es baja, PCA aporta poco.

*PCA vs. Análisis factorial*: ambas técnicas reducen variables correlacionadas a un número menor de dimensiones, pero con propósitos distintos. PCA busca **resumir varianza** (útil para preparar variables antes de una regresión o un análisis de conglomerados). El análisis factorial busca **identificar constructos latentes** que explican por qué las variables están correlacionadas (útil para diseñar y validar escalas de cuestionario, como en el notebook 21).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

El archivo `autos.xlsx` contiene las especificaciones técnicas de 15 modelos de automóviles: desplazamiento del motor, momento (torque), potencia, dimensiones, peso, capacidad de cajuela, velocidad máxima y tiempo de aceleración de 0 a 100 km/h.

In [ ]:
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/autos.xlsx')
df.head()

In [ ]:
var_num = ['desplazamiento', 'momento', 'potencia', 'longitud', 'ancho',
           'peso', 'cajuela', 'velocidad', 'aceleracion']
df[var_num].describe().T

Antes de aplicar PCA, revisemos la matriz de correlaciones: es precisamente la alta correlación entre estas variables la que hace útil reducirlas a unos pocos componentes.

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(df[var_num].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlaciones');

**Estandarizar variables**: al igual que en el análisis de conglomerados, es indispensable llevar todas las variables a la misma escala antes de aplicar PCA.

In [ ]:
scaler = StandardScaler()
X_std = scaler.fit_transform(df[var_num])

## Extracción de componentes

Ajustamos PCA sin limitar el número de componentes para revisar cuánta varianza explica cada uno.

In [ ]:
pca = PCA()
pca.fit(X_std)

varianza_explicada = pd.DataFrame({
    'Componente': [f'PC{i+1}' for i in range(len(var_num))],
    'Autovalor (eigenvalue)': pca.explained_variance_,
    'Varianza explicada (%)': pca.explained_variance_ratio_ * 100,
    'Varianza acumulada (%)': np.cumsum(pca.explained_variance_ratio_) * 100
})
varianza_explicada.round(2)

## Determinación del número de componentes

Al igual que en análisis factorial, se puede utilizar el **criterio de Kaiser** (retener componentes con autovalor mayor a 1) o el **gráfico de sedimentación (scree plot)**, buscando el punto donde la curva se vuelve horizontal.

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(var_num) + 1), pca.explained_variance_, marker='o')
plt.axhline(1, color='red', linestyle='--', label='Criterio de Kaiser (autovalor = 1)')
plt.xlabel('Componente')
plt.ylabel('Autovalor (eigenvalue)')
plt.title('Gráfico de sedimentación (scree plot)')
plt.legend()
plt.show()

El primer componente concentra por sí solo más del 80% de la varianza, y junto con el segundo superan el 94%. Ambos autovalores están por encima de 1, mientras que el resto cae por debajo — tanto el criterio de Kaiser como el scree plot sugieren retener **2 componentes**.

In [ ]:
pca = PCA(n_components=2)
componentes = pca.fit_transform(X_std)

print(f'Varianza explicada por los 2 componentes: {pca.explained_variance_ratio_.sum():.1%}')

**Cargas (loadings)**: indican qué tanto correlaciona cada variable original con cada componente. Cargas con el mismo signo y magnitud alta ayudan a interpretar qué representa cada componente.

In [ ]:
cargas = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=var_num)
cargas.round(3)

*Interpretación*: todas las variables cargan positivamente en PC1 excepto la aceleración (tiempo en segundos de 0 a 100 km/h), que carga en negativo — un auto más grande, pesado y potente tarda **menos** tiempo en acelerar. PC1 puede interpretarse como un índice general de "tamaño y potencia del vehículo". En PC2, `cajuela` y `longitud` tienen las cargas más altas y positivas, mientras que `potencia` carga en negativo — PC2 contrasta autos espaciosos/familiares contra autos compactos y deportivos, independientemente de PC1.

In [ ]:
scores = pd.DataFrame(componentes, columns=['PC1', 'PC2'])
scores['Auto'] = df['Name'].values

plt.figure(figsize=(8, 6))
sns.scatterplot(data=scores, x='PC1', y='PC2')
for i in range(len(scores)):
    plt.text(scores['PC1'][i] + 0.1, scores['PC2'][i], scores['Auto'][i], fontsize=8)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.title('Autos en el espacio de los dos primeros componentes')
plt.show()

**Ejemplo de reporte de resultados**:
>"Se aplicó un análisis de componentes principales a 9 especificaciones técnicas de 15 modelos de automóviles, previa estandarización de las variables. Los dos primeros componentes explicaron el 94.4% de la varianza total (81.7% y 12.7%, respectivamente), superando ambos el criterio de Kaiser (autovalor > 1). El primer componente se interpretó como un índice general de tamaño y potencia del vehículo, con cargas positivas en todas las variables de desempeño y dimensiones, y carga negativa en el tiempo de aceleración. El segundo componente contrastó vehículos espaciosos (cajuela, longitud) frente a vehículos compactos y deportivos. En el plano de ambos componentes, los autos económicos (Kia Picanto, Suzuki Splash, Renault Clio) se agruparon en valores bajos de PC1, mientras que los deportivos y de lujo (Porsche Cayman, Mercedes E280, BMW 525i) se ubicaron en valores altos, consistente con la interpretación del componente."

## Uso posterior de los componentes

Los puntajes de los componentes (`scores`) son variables numéricas no correlacionadas entre sí, por lo que pueden utilizarse directamente como insumo en otras técnicas, evitando el problema de multicolinealidad de las variables originales:
- Como predictores en un modelo de regresión (notebooks 16-18), en lugar de las 9 variables originales altamente correlacionadas.
- Como variables de entrada para un análisis de conglomerados (notebook 19), en lugar de seleccionar manualmente un subconjunto de variables no correlacionadas.

## Ejercicio

En el notebook 19 (Segmentación de mercados), el análisis de conglomerados con el archivo `cerveza.xlsx` excluyó la variable `Alcohol` porque estaba altamente correlacionada con `Calorias` (r = 0.91), y se utilizó únicamente `Calorias` y `Costo100ml`.

1. Aplica PCA a las 4 variables cuantitativas de `cerveza.xlsx` (`Calorias`, `Alcohol`, `Contenido`, `Costo100ml`), sin excluir ninguna.
2. ¿Cuántos componentes retendrías según el criterio de Kaiser?
3. Compara este enfoque con la decisión manual de excluir `Alcohol`: ¿qué ventaja tiene dejar que PCA resuma la información en lugar de eliminar una variable?

In [ ]:
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/cerveza.xlsx')

## Referencias
- Documentación de `sklearn.decomposition.PCA`: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html